# Program 11: Connection Flood Demonstration – Security Lab

## Aim:
Write a Python program to demonstrate, in a controlled local security lab, how repeated connection attempts can increase server workload and explain the importance of rate limiting.

## Concepts Covered:
* High-volume socket generation loops
* Performance and overhead analysis using execution timers
* Mitigation strategies (Rate Limiting, Firewalls, Connection Throttling)


In [1]:
import socket
import threading
import time

def run_target_server(host, port):
    """A simple server that counts connections to demonstrate increasing workload."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
            server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            server_socket.bind((host, port))
            server_socket.listen(100) # Accept backlog queue up to 100
            
            connection_count = 0
            while True:
                # Accept incoming simulation connections
                client_conn, addr = server_socket.accept()
                connection_count += 1
                
                # Instantly close connection to process the next one quickly
                client_conn.close()
                
    except Exception as e:
        pass  # Quietly catch background resource cleanup exceptions

def simulate_connection_flood(host, port, target_requests=50):
    """Simulates rapid sequential connection attempts to monitor performance metrics."""
    print(f"=== SECURITY LAB: CONNECTION FLOOD SIMULATION ===")
    print(f"Targeting Local Test Environment -> {host}:{port}")
    print(f"Dispatched Load Volume: {target_requests} sequential connections\n")
    
    start_time = time.time()
    successful_connections = 0
    
    # Rapid fire loop to trigger connections
    for i in range(1, target_requests + 1):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client_socket:
                client_socket.settimeout(0.5)  # Short timeout ceiling per connection
                client_socket.connect((host, port))
                successful_connections += 1
        except (socket.error, socket.timeout):
            # Track if connections drop due to resource exhaustion
            pass
            
    elapsed_time = time.time() - start_time
    
    # Calculate performance overhead matrix
    print("=" * 55)
    print(f"SIMULATION COMPLETION METRICS")
    print("=" * 55)
    print(f"Total Requests Dispatched:   {target_requests}")
    print(f"Successful Connections:      {successful_connections}/{target_requests}")
    print(f"Total Processing Time:       {elapsed_time:.4f} seconds")
    print(f"Average Latency Per Request: {(elapsed_time / target_requests) * 1000:.2f} ms")
    print("=" * 55)
    
    # Educational security summary callout
    print("\n[Security Concept Education Note]")
    print("Without defense layers, each connection forces the operating system to allocate")
    print("file descriptors, memory buffers, and CPU cycles. To mitigate connection fatigue,")
    print("production applications rely on Rate Limiting (e.g., token bucket algorithms),")
    print("reverse proxy firewalls, or connection-per-IP limits to drop flood packets early.")

if __name__ == "__main__":
    HOST = '127.0.0.1'
    PORT = 65440  # Alternative high port to minimize WinError access conflicts
    
    # 1. Spin up the targeted test server in an independent background thread
    lab_server = threading.Thread(target=run_target_server, args=(HOST, PORT), daemon=True)
    lab_server.start()
    
    # 2. Allow a brief split-second window for port binding allocation
    time.sleep(0.3)
    
    # 3. Fire up the controlled high-frequency simulator engine
    simulate_connection_flood(HOST, PORT, target_requests=50)


=== SECURITY LAB: CONNECTION FLOOD SIMULATION ===
Targeting Local Test Environment -> 127.0.0.1:65440
Dispatched Load Volume: 50 sequential connections

SIMULATION COMPLETION METRICS
Total Requests Dispatched:   50
Successful Connections:      50/50
Total Processing Time:       0.3730 seconds
Average Latency Per Request: 7.46 ms

[Security Concept Education Note]
Without defense layers, each connection forces the operating system to allocate
file descriptors, memory buffers, and CPU cycles. To mitigate connection fatigue,
production applications rely on Rate Limiting (e.g., token bucket algorithms),
reverse proxy firewalls, or connection-per-IP limits to drop flood packets early.
